# Pre-config that could be necessary. 
## Please read with attention

```bash
uv pip install "numpy<2"

uv pip install pandana
sudo apt install build-essential g++ python3-dev
sudo apt install gdal-bin libgdal-dev libgeos-dev libproj-dev
```

I am running this as jupyter notebook. You can use it the way you like it, Vscode also has a jupyter extension. 
To keep enviroment consistency, I did the following:
1. Added [project.optional-dependencies] to project.toml (it's on the main branch, no need to add anything yourself)
2. Now we can switch to a notebook env:
   ```bash
   uv sync --extra notebooks
   # and then switch back if we don't need jupyter anymore for our application
   uv synv
   ```
3. then go to the ./Notebooks/ folder and run it with:
   ```bash
   jupyter notebook
   ```
4. If your browser does not open, just copy and paste the URL provided on the terminal


**Unfortunally, Github did not showed the maps and the graphs, so I had to clean the cells to at least show the code.**

Before running with new data:

- [ ] GeoJSON has polygon geometries
- [ ] GeoJSON has a district name column — update DISTRICT_NAME
- [ ] GeoJSON has a numeric socioeconomic column — update SOCIO_INDEX
- [ ] Socioeconomic column has no all-zero variance — otherwise normalization fails
- [ ] POI_TYPE exists in POI_TAGS — extend the dictionary if needed

In [ ]:
# ============================================================
# Who Can Reach What
# Urban Accessibility Analysis for Spatial Justice
# Münster Case Study
# <signed-off-by fillfeitosa@gmai.com>
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
import pandana
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path



# --- Configuration ---
DATA_FILE        = "data.geojson"
SOCIO_INDEX      = "pct_welfare_15_64"
DISTRICT_NAME    = "district_name"
POI_TYPE         = "bus_stop"
MAX_DISTANCE     = 1000   # meters — walking ~12 minutes
NUM_POIS         = 5      # count opportunities within MAX_DISTANCE
NETWORK_TYPE     = "walk"

# Overpass configurations, not normally needed
ox.settings.overpass_url = "https://overpass-api.de/api/interpreter"

ox.settings.requests_timeout = 180
ox.settings.use_cache = True
ox.settings.log_console = True


Path("reports").mkdir(exist_ok=True)
print("Configuration ready")

In [ ]:
# ============================================================
# STEP 1 — Load and validate the GeoDataFrame
# ============================================================

def load_geodata(filename: str) -> gpd.GeoDataFrame:
    """
    Loads a GeoJSON file and reprojects to EPSG:25832
    (UTM zone 32N — metric CRS standard for Germany).
    
    EPSG:25832 is used for all spatial operations because
    it preserves metric distances and areas accurately.
    
    Args:
        filename: path to GeoJSON file
        
    Returns:
        GeoDataFrame in EPSG:25832
    """
    path = Path(filename)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    
    gdf = gpd.read_file(path)
    gdf = gdf.to_crs(epsg=25832)
    print(f"Loaded {len(gdf)} districts from {filename}")
    return gdf


gdf = load_geodata(DATA_FILE)
gdf[[DISTRICT_NAME, SOCIO_INDEX, "geometry"]].head()

In [ ]:
# ============================================================
# STEP 2 — Extract bounding box in WGS84
# ============================================================

def get_bbox_wgs84(gdf: gpd.GeoDataFrame) -> tuple:
    """
    Converts the GeoDataFrame extent to WGS84 and returns
    the bounding box as (west, south, east, north).
    
    osmnx requires lat/lon coordinates — we convert here
    once and reuse the result for all OSM downloads.
    The GeoDataFrame itself stays in EPSG:25832.
    
    Args:
        gdf: GeoDataFrame in any projected CRS
        
    Returns:
        (west, south, east, north) in decimal degrees
    """
    gdf_wgs84 = gdf.to_crs(epsg=4326)
    west, south, east, north = gdf_wgs84.total_bounds
    print(f"Bounding box — W:{west:.4f} S:{south:.4f} E:{east:.4f} N:{north:.4f}")
    return west, south, east, north


bbox = get_bbox_wgs84(gdf)

In [ ]:
# ============================================================
# STEP 3 — Download OSM street network and build pandana graph
# ============================================================

def download_network(bbox: tuple, network_type: str = "walk"):
    """
    Downloads the OSM street network for the bounding box.
    
    Args:
        bbox:         (north, south, east, west) in WGS84
        network_type: 'walk', 'drive', or 'bike'
        
    Returns:
        osmnx MultiDiGraph
    """
    print(f"Downloading OSM {network_type} network...")
    graph = ox.graph_from_bbox(
        bbox,
        network_type=network_type
    )
    print(f"Network ready — {len(graph.nodes)} nodes, {len(graph.edges)} edges")
    return graph


def build_pandana_network(graph) -> pandana.Network:
    """
    Converts an osmnx graph to a pandana Network.
    Edge weights are lengths in meters as computed by osmnx.
    
    Args:
        graph: osmnx MultiDiGraph
        
    Returns:
        pandana.Network ready for accessibility queries
    """
    nodes, edges = ox.graph_to_gdfs(graph, nodes=True, edges=True)
    
    edge_from    = pd.Series(edges.index.get_level_values("u").values)
    edge_to      = pd.Series(edges.index.get_level_values("v").values)
    edge_weights = pd.DataFrame({"distance": edges["length"].values})
    
    network = pandana.Network(
        node_x=nodes["x"],
        node_y=nodes["y"],
        edge_from=edge_from,
        edge_to=edge_to,
        edge_weights=edge_weights,
        twoway=False
    )
    print(f"Pandana network ready — {len(nodes)} nodes")
    return network


graph   = download_network(bbox, network_type=NETWORK_TYPE)
# It's also possible to crete graphs from named places, like:
# graph = ox.graph_from_place(
#     "Münster, Germany",
#     network_type="bike"
# )
network = build_pandana_network(graph)

In [ ]:
ox.save_graphml(graph, "muenster_walk.graphml") # We can save and load our network if needed

In [ ]:
# ============================================================
# STEP 4 — Download POIs and register on pandana network
# ============================================================

POI_TAGS = {
    "school":            {"amenity": "school"},
    "kindergarten":      {"amenity": "kindergarten"},
    "bus_stop":          {"highway": "bus_stop"},
    "hospital":          {"amenity": "hospital"},
    "clinic":            {"amenity": "clinic"},
    "pharmacy":          {"amenity": "pharmacy"},
    "supermarket":       {"shop": "supermarket"},
    "park":              {"leisure": "park"},
    "employment_center": {"office": "employment_agency"},
}


def download_pois(bbox: tuple, poi_type: str) -> gpd.GeoDataFrame:
    """
    Downloads POIs from OSM for the given category.
    Polygon geometries (e.g. school buildings) are
    converted to centroids for pandana compatibility.
    
    Args:
        bbox:     (north, south, east, west) in WGS84
        poi_type: key from POI_TAGS dictionary
        
    Returns:
        GeoDataFrame of POI point geometries in WGS84
    """
    if poi_type not in POI_TAGS:
        raise ValueError(
            f"Unknown POI type '{poi_type}'. "
            f"Available: {list(POI_TAGS.keys())}"
        )
    
    
    tags = POI_TAGS[poi_type]
    
    print(f"Downloading {poi_type} POIs from OSM...")
    pois = ox.features_from_bbox(
        bbox=bbox,
        tags=tags
    )
    pois = pois.to_crs(epsg=4326)
    pois["geometry"] = pois.geometry.centroid
    print(f"Found {len(pois)} {poi_type} locations")
    return pois


def register_pois(
    network:      pandana.Network,
    pois:         gpd.GeoDataFrame,
    poi_type:     str,
    max_distance: float = MAX_DISTANCE,
    max_items:    int   = NUM_POIS,
) -> None:
    """
    Registers POI locations on the pandana network.
    Must be called before compute_accessibility.
    
    max_distance and max_items here must match what
    is later passed to nearest_pois — they define the
    search space pandana prepares internally.
    
    Args:
        network:      pandana Network object
        pois:         GeoDataFrame of POI points in WGS84
        poi_type:     category name matching POI_TAGS
        max_distance: maximum search radius in meters
        max_items:    maximum number of POIs to return per node
    """
    network.set_pois(
        category=poi_type,
        maxdist=max_distance,
        maxitems=max_items,
        x_col=pois.geometry.x,
        y_col=pois.geometry.y,
    )
    print(f"{poi_type} POIs registered on network")


pois = download_pois(bbox, POI_TYPE)
register_pois(network, pois, POI_TYPE)
pois.head(3)

In [ ]:
bbox

In [ ]:
# ============================================================
# STEP 5 — Compute accessibility
# ============================================================

def compute_accessibility(
    network:      pandana.Network,
    poi_type:     str,
    max_distance: float = MAX_DISTANCE,
    num_pois:     int   = NUM_POIS,
) -> pd.DataFrame:
    """
    Computes two accessibility metrics for every node in the network:

    1. dist_1...dist_N: distance in meters to the Nth nearest POI.
       Nodes with no POI within max_distance receive max_distance
       as the value. Lower is better.

    2. opportunities: count of POIs reachable within max_distance.
       Derived from the distance columns — any dist_N < max_distance
       counts as one reachable opportunity. Higher is better.
       This is a meaningful spatial justice metric because
       it asks not just "how far is the nearest opportunity" but
       "how many opportunities can this neighborhood reach?"

    Args:
        network:      pandana Network with POIs already registered
        poi_type:     category name used in register_pois
        max_distance: search radius in meters
        num_pois:     number of nearest POIs to compute distances for

    Returns:
        DataFrame indexed by node_id with columns:
            dist_1 ... dist_N  — distance to Nth nearest POI
            opportunities      — count of POIs within max_distance
    """
    print(f"Precomputing network reachability up to {max_distance}m...")
    network.precompute(max_distance)

    distances = network.nearest_pois(
        distance=max_distance,
        category=poi_type,
        num_pois=num_pois,
        max_distance=max_distance,
    )
    distances.columns = [f"dist_{i}" for i in range(1, num_pois + 1)]

    # count POIs within max_distance — any distance strictly below
    # max_distance is a reachable opportunity
    distances["opportunities"] = (distances < max_distance).sum(axis=1)

    print(f"Accessibility computed for {len(distances)} network nodes")
    return distances


accessibility = compute_accessibility(network, POI_TYPE)
accessibility.head()

In [ ]:
# ============================================================
# STEP 6 — Aggregate node-level accessibility to districts
# ============================================================

def aggregate_to_districts(
    gdf:           gpd.GeoDataFrame,
    graph,
    accessibility: pd.DataFrame,
    district_col:  str = DISTRICT_NAME,
) -> gpd.GeoDataFrame:
    """
    Aggregates node-level accessibility values to district polygons.

    For each district we compute:
        mean_dist   — average distance to nearest POI across all nodes
        min_dist    — best access point in the district
        max_dist    — worst access point in the district
        std_dist    — internal inequality of access within the district
        mean_opps   — average opportunity count (POIs reachable per node)

    A large std_dist means the district has internally unequal access —
    some residents are well served, others are not. This is invisible
    if you only report mean_dist.

    The spatial join uses EPSG:4326 because osmnx node coordinates
    are in WGS84. The result is reprojected back to EPSG:25832.

    Args:
        gdf:           GeoDataFrame of district polygons in EPSG:25832
        graph:         osmnx graph (used to retrieve node coordinates)
        accessibility: DataFrame from compute_accessibility
        district_col:  column name for district labels

    Returns:
        GeoDataFrame with one row per district and accessibility columns
    """
    nodes = ox.graph_to_gdfs(graph, nodes=True, edges=False)
    
    nodes_gdf = gpd.GeoDataFrame(
        accessibility,
        geometry=gpd.points_from_xy(nodes["x"], nodes["y"]),
        crs="EPSG:4326"
    )

    gdf_wgs84 = gdf[[district_col, "geometry"]].to_crs(epsg=4326)
    joined    = gpd.sjoin(nodes_gdf, gdf_wgs84, how="left", predicate="within")

    grouped = joined.groupby(district_col).agg(
        mean_dist =("dist_1", "mean"),
        min_dist  =("dist_1", "min"),
        max_dist  =("dist_1", "max"),
        std_dist  =("dist_1", "std"),
        mean_opps =("opportunities", "mean"),
    ).reset_index()

    result = gdf.merge(grouped, on=district_col, how="left")

    print(f"Aggregated accessibility for {grouped[district_col].nunique()} districts")
    return result


gdf_access = aggregate_to_districts(gdf, graph, accessibility)
gdf_access[[DISTRICT_NAME, "mean_dist", "min_dist", "max_dist", "std_dist", "mean_opps"]].head()

In [ ]:
# ============================================================
# STEP 7 — Report: rankings and socioeconomic comparison
# ============================================================

def compute_compound_deprivation(
    gdf:          gpd.GeoDataFrame,
    socio_col:    str = SOCIO_INDEX,
    access_col:   str = "mean_dist",
    district_col: str = DISTRICT_NAME,
    top_n:        int = 5,
    invert_socio:  bool = False,
) -> pd.DataFrame:
    """
    Identifies districts experiencing compound deprivation —
    simultaneously high poverty AND poor transit access.

    Both metrics are normalized to [0,1] so they contribute
    equally to the compound score regardless of their original
    scale. A district scoring high on both is genuinely
    doubly disadvantaged.
    
    invert_socio: set True if lower values mean more deprivation
                  e.g. an income index where low = poor
    """
    data = gdf[[district_col, socio_col, access_col]].dropna().copy()

    # normalize both to [0, 1]
    for col in [socio_col, access_col]:
        data[f"{col}_norm"] = (
            (data[col] - data[col].min()) /
            (data[col].max() - data[col].min())
        )

    # invert if needed so high normalized score always means more deprived
    if invert_socio:
        data[f"{socio_col}_norm"] = 1 - data[f"{socio_col}_norm"]
    
    # compound score: equal weight on poverty and poor access
    data["compound_score"] = (
        data[f"{socio_col}_norm"] + data[f"{access_col}_norm"]
    ) / 2

    return data.nlargest(top_n, "compound_score")[
        [district_col, socio_col, access_col, "compound_score"]
    ]



def report_rankings(
    gdf:          gpd.GeoDataFrame,
    socio_col:    str = SOCIO_INDEX,
    district_col: str = DISTRICT_NAME,
    top_n:        int = 5,
) -> pd.DataFrame:
    """
    Produces a ranking table of districts by mean accessibility.

    Reports the top_n best and top_n worst districts by mean
    distance to nearest POI, alongside their socioeconomic index.

    A district that ranks poorly on BOTH mean_dist AND socio_col
    is experiencing compound deprivation — poor access AND high
    poverty. That is the spatial justice signal this report
    is designed to surface.

    Args:
        gdf:          GeoDataFrame with accessibility and socio columns
        socio_col:    socioeconomic index column name
        district_col: district label column
        top_n:        number of best/worst districts to show

    Returns:
        DataFrame with full ranking
    """
    cols   = [district_col, "mean_dist", "min_dist",
              "max_dist", "std_dist", "mean_opps", socio_col]
    ranked = gdf[cols].dropna().sort_values("mean_dist").reset_index(drop=True)
    ranked["rank"] = ranked.index + 1

    print("=" * 65)
    print(f"  Accessibility Ranking — {POI_TYPE.upper()} within {MAX_DISTANCE}m")
    print("=" * 65)

    print(f"\n--- TOP {top_n} BEST ACCESS ---")
    print(ranked.head(top_n)[[district_col, "mean_dist", "mean_opps", socio_col]].to_string(index=False))

    print(f"\n--- TOP {top_n} WORST ACCESS ---")
    print(ranked.tail(top_n)[[district_col, "mean_dist", "mean_opps", socio_col]].to_string(index=False))

    print("\n--- COMPOUND DEPRIVATION (worst access + highest poverty) ---")
    # compound = ranked.nlargest(top_n, socio_col)[
    #     [district_col, "mean_dist", "mean_opps", socio_col]
    # ]
    compound = compute_compound_deprivation(gdf_access)
    print(compound.to_string(index=False))
    print("=" * 65)

    ranked.to_csv(f"reports/accessibility_ranking_{POI_TYPE}.csv", index=False)
    print(f"\nFull ranking saved to reports/accessibility_ranking_{POI_TYPE}.csv")
    return ranked


ranked = report_rankings(gdf_access)

In [ ]:
def compute_compound_deprivation(
    gdf:          gpd.GeoDataFrame,
    socio_col:    str = SOCIO_INDEX,
    access_col:   str = "mean_dist",
    district_col: str = DISTRICT_NAME,
    top_n:        int = 5,
) -> pd.DataFrame:
    """
    Identifies districts experiencing compound deprivation —
    simultaneously high poverty AND poor transit access.

    Both metrics are normalized to [0,1] so they contribute
    equally to the compound score regardless of their original
    scale. A district scoring high on both is genuinely
    doubly disadvantaged.
    """
    data = gdf[[district_col, socio_col, access_col]].dropna().copy()

    # normalize both to [0, 1]
    for col in [socio_col, access_col]:
        data[f"{col}_norm"] = (
            (data[col] - data[col].min()) /
            (data[col].max() - data[col].min())
        )

    # compound score: equal weight on poverty and poor access
    data["compound_score"] = (
        data[f"{socio_col}_norm"] + data[f"{access_col}_norm"]
    ) / 2

    return data.nlargest(top_n, "compound_score")[
        [district_col, socio_col, access_col, "compound_score"]
    ]


compound = compute_compound_deprivation(gdf_access)
print(compound.to_string(index=False))

In [ ]:
# ============================================================
# STEP 8 — Visualization
# ============================================================

def plot_accessibility_map(
    gdf:          gpd.GeoDataFrame,
    column:       str   = "mean_dist",
    district_col: str   = DISTRICT_NAME,
    title:        str   = None,
    cmap:         str   = "RdYlGn_r",
) -> plt.Figure:
    """
    Choropleth map of an accessibility metric across districts.
    RdYlGn_r: red = poor access (high distance), green = good access.
    """
    if title is None:
        title = f"{column} to nearest {POI_TYPE} (meters)"

    fig, ax = plt.subplots(figsize=(10, 10))
    gdf.plot(
        column=column,
        cmap=cmap,
        legend=True,
        legend_kwds={"label": "meters", "shrink": 0.6},
        edgecolor="white",
        linewidth=0.5,
        ax=ax,
    )
    # label each district
    for _, row in gdf.iterrows():
        if pd.notna(row[column]):
            ax.annotate(
                row[district_col],
                xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                fontsize=5,
                ha="center",
                color="black",
            )
    ax.set_title(title, fontsize=13, pad=12)
    ax.set_axis_off()
    return fig


def plot_access_vs_socio(
    gdf:          gpd.GeoDataFrame,
    socio_col:    str = SOCIO_INDEX,
    district_col: str = DISTRICT_NAME,
) -> plt.Figure:
    """
    Scatter plot of mean accessibility distance vs socioeconomic index.

    Each point is a district. Points in the top-right quadrant
    (high distance + high poverty) are experiencing compound
    deprivation — the core spatial justice finding.
    """
    data = gdf[[district_col, "mean_dist", socio_col]].dropna()

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(
        data["mean_dist"],
        data[socio_col],
        color="steelblue",
        alpha=0.7,
        s=60,
        edgecolors="white",
        linewidth=0.5,
    )

    # label each point
    for _, row in data.iterrows():
        ax.annotate(
            row[district_col],
            xy=(row["mean_dist"], row[socio_col]),
            fontsize=7,
            xytext=(4, 4),
            textcoords="offset points",
        )

    # quadrant lines at medians
    ax.axvline(data["mean_dist"].median(), color="grey",
               linestyle="--", linewidth=0.8, alpha=0.6)
    ax.axhline(data[socio_col].median(), color="grey",
               linestyle="--", linewidth=0.8, alpha=0.6)

    # annotate quadrants
    xmax = data["mean_dist"].max()
    ymax = data[socio_col].max()
    ax.text(xmax * 0.98, ymax * 0.98, "Compound\ndeprivation",
            ha="right", va="top", fontsize=8,
            color="red", alpha=0.7)

    ax.set_xlabel(f"Mean distance to nearest {POI_TYPE} (meters)", fontsize=11)
    ax.set_ylabel(f"{socio_col} (%)", fontsize=11)
    ax.set_title(
        f"Accessibility vs Socioeconomic Index\n"
        f"{POI_TYPE.capitalize()} within {MAX_DISTANCE}m — Münster",
        fontsize=12
    )
    return fig


# --- run visualizations ---
fig_map = plot_accessibility_map(gdf_access, column="mean_dist")
fig_map.savefig(f"reports/map_mean_dist_{POI_TYPE}.png", dpi=150, bbox_inches="tight")

fig_map_opps = plot_accessibility_map(
    gdf_access, column="mean_opps",
    cmap="RdYlGn",
    title=f"Mean opportunities — {POI_TYPE} within {MAX_DISTANCE}m"
)
fig_map_opps.savefig(f"reports/map_opportunities_{POI_TYPE}.png", dpi=150, bbox_inches="tight")

fig_scatter = plot_access_vs_socio(gdf_access)
fig_scatter.savefig(f"reports/scatter_access_vs_socio_{POI_TYPE}.png", dpi=150, bbox_inches="tight")

plt.show()
print("All reports saved to reports/")
network.plot(accessibility["dist_1"]) # but this is litmited. Let's create a function to better print the accessbility

In [ ]:
def plot_network_accessibility(
    gdf:           gpd.GeoDataFrame,
    graph,
    accessibility: pd.DataFrame,
    column:        str = "dist_1",
    title:         str = None,
    cmap:          str = "RdYlGn_r",
) -> plt.Figure:
    """
    Plots node-level accessibility as a colored scatter overlay
    on top of the district polygon boundaries.

    Each dot is a network node colored by its accessibility value.
    RdYlGn_r: red = far from POI (poor access), green = close (good access).
    The district boundaries give spatial reference without dominating the map.

    Args:
        gdf:           GeoDataFrame of district polygons
        graph:         osmnx graph — used to retrieve node coordinates
        accessibility: DataFrame from compute_accessibility
        column:        which column to visualize (dist_1, opportunities, etc.)
        title:         plot title — auto-generated if None
        cmap:          matplotlib colormap

    Returns:
        matplotlib Figure
    """

    nodes  = ox.graph_to_gdfs(graph, nodes=True, edges=False)
    values = accessibility[column].reindex(nodes.index)

    # clip nodes to the GeoDataFrame boundary
    # routing used the full network — display only what's inside
    boundary       = gdf.to_crs(epsg=4326).union_all()
    nodes_clipped  = nodes[nodes.geometry.within(boundary)]
    values_clipped = values.reindex(nodes_clipped.index)

    if title is None:
        label = "Distance to nearest POI (m)" if "dist" in column else "Opportunities"
        title = f"Network Accessibility — {label}\n{POI_TYPE.capitalize()} within {MAX_DISTANCE}m — Münster"

    fig, ax = plt.subplots(figsize=(12, 12))

    gdf.to_crs(epsg=4326).plot(
        ax=ax,
        color="lightgrey",
        edgecolor="white",
        linewidth=0.8,
        alpha=0.6,
    )

    sc = ax.scatter(
        nodes_clipped["x"],
        nodes_clipped["y"],
        c=values_clipped,
        cmap=cmap,
        s=3,
        alpha=0.7,
        linewidths=0,
    )

    plt.colorbar(
        sc, ax=ax,
        label="meters" if "dist" in column else "count",
        shrink=0.5,
        pad=0.01,
    )

    ax.set_title(title, fontsize=13, pad=12)
    ax.set_axis_off()
    return fig

# --- distance to nearest POI ---
fig_net = plot_network_accessibility(
    gdf, graph, accessibility,
    column="dist_1",
)
fig_net.savefig(f"reports/network_dist_{POI_TYPE}.png", dpi=150, bbox_inches="tight")

# --- opportunity count ---
fig_opps = plot_network_accessibility(
    gdf, graph, accessibility,
    column="opportunities",
    cmap="RdYlGn",
    title=f"Opportunities — {POI_TYPE.capitalize()} within {MAX_DISTANCE}m — Münster",
)
fig_opps.savefig(f"reports/network_opps_{POI_TYPE}.png", dpi=150, bbox_inches="tight")

plt.show()